# 第 1 周末练习 —— 技术问答解释器（多后端流式）

## 练习目标（理念）

为了展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一段需要解释的代码 / 技术问题
- **输出**：清晰、耐心的分步说明（含「做什么 / 为什么 / 简单示例」）
- **额外要求**：用**流式（streaming）**一边生成一边更新 Markdown 显示

这是你在课程期间自己也能天天用的工具：把看不懂的代码丢进来，对比云端 GPT 与本地 / 云端 Ollama 模型的回答。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `client.chat.completions.create(...)` |
| `messages`（system / user） | `system_prompt` 定风格，`user_prompt` 放题目 |
| 流式输出 `stream=True` | 逐块拼接，用 `update_display` 刷新 |
| OpenAI 云端模型 | `MODEL_GPT`（默认 `gpt-4o-mini`） |
| Ollama 本地端点 | 同一个 `base_url`，换不同 `model` 名（本地 / cloud） |

## 怎么跑

1. 按下一格说明写好 `.env`，并确保本机 Ollama 已启动
2. 从上到下依次运行每个单元格（Shift+Enter）
3. 在「提问」单元格改 `question`，再分别跑三个 `stream_response(...)` 对比结果


In [1]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 API Key、模型名、Ollama 地址
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display / update_display 做流式刷新
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：既可打云端 API，也可指向 Ollama 的 OpenAI 兼容端点
from openai import OpenAI


## 环境设置

在运行本笔记本之前，请把下面这些键值写进项目根目录的 `.env`（名称必须一致，值换成你自己的）：

```env
OPENAI_API_KEY=your_openai_key
OPENAI_MODEL=gpt-4o-mini

OLLAMA_API_KEY=ollama
OLLAMA_BASE_URL=http://localhost:11434/v1
OLLAMA_MODEL=llama3.2
OLLAMA_CLOUD_MODEL=gpt-oss:20b-cloud
```

说明：

- 两个 Ollama 客户端都指向**同一个本地端点**（`OLLAMA_BASE_URL`，OpenAI 兼容的 `/v1`）。
- 区别只在**模型名**：`OLLAMA_MODEL`（本机模型）与 `OLLAMA_CLOUD_MODEL`（经 Ollama 访问的 cloud 模型名）。
- `OPENAI_API_KEY` 给云端 OpenAI 用；Ollama 侧常用占位密钥 `ollama`（本地一般不校验真伪）。


In [2]:
# ========== 常量：模型名与默认 Ollama 地址集中管理 ==========

# 从环境变量读 OpenAI 模型名；若未设置则回退到 gpt-4o-mini
MODEL_GPT = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
# 本地 Ollama 模型名（需事先 ollama pull）；字符串须与本机已安装模型一致
MODEL_LLAMA = os.getenv('OLLAMA_MODEL', 'llama3.2')
# 经同一本地端点调用的「cloud」模型名（具体可用性取决于你的 Ollama / 账号配置）
MODEL_LLAMA_CLOUD = os.getenv('OLLAMA_CLOUD_MODEL', 'gpt-oss:20b-cloud')
# 默认的 Ollama OpenAI 兼容基址；后面若 .env 有 OLLAMA_BASE_URL 会覆盖使用
OLLAMA_BASE_URL = 'http://localhost:11434/v1'


In [3]:
# ========== 环境 + 客户端：一份密钥检查，三个 OpenAI 兼容客户端 ==========

# 加载 .env：override=True 表示用文件里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 读取 OpenAI 云端 API Key（常见变量名 OPENAI_API_KEY）
api_key = os.getenv('OPENAI_API_KEY')
# Ollama 侧密钥：没有就用占位字符串 'ollama'（本地兼容端点通常需要某个非空 api_key）
api_key_ollama = os.getenv('OLLAMA_API_KEY') or 'ollama'
# Ollama 基址：优先 .env；否则用上一格常量 OLLAMA_BASE_URL
ollama_base_url = os.getenv('OLLAMA_BASE_URL') or OLLAMA_BASE_URL

# 对 OPENAI 密钥做一次「看起来像不像」的健全性检查（sanity check），不保证一定可用
if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print('API key looks good so far')
else:
    print('There might be a problem with your API key? Please visit the troubleshooting notebook!')

# 创建默认 OpenAI 客户端：会自动从环境变量读 OPENAI_API_KEY，打官方云端
openai = OpenAI()

# 创建指向本地 Ollama /v1 的客户端（后面本地模型与 cloud 模型名都走这个端点族）
print(f'Using local Ollama endpoint for both models: {ollama_base_url}')
# 本地模型客户端：base_url 指向 Ollama，api_key 用上面的占位/配置
ollama = OpenAI(base_url=ollama_base_url, api_key=api_key_ollama)
# 再做一个同端点客户端变量：后面用不同 model 名调用（逻辑上可与 ollama 共用，这里保持原作者写法）
ollama_cloud = OpenAI(base_url=ollama_base_url, api_key=api_key_ollama)


API key looks good so far
Using local Ollama endpoint for both models: http://localhost:11434/v1


In [4]:
# ========== 提问与提示词：改 question 就能问新问题 ==========

# 用户真正关心的题目（发给模型的内容保持英文：影响回答的字符串不翻译）
# 练习建议：换成你自己今天看不懂的几行代码，再分别跑后面三个流式单元格对比
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

# system prompt：约定助手人设与回答结构（保留英文，改译会改变模型行为/风格）
system_prompt = """
You are a clear and patient coding assistant. When given a piece of code, you will:
1. Explain what the code does, step by step.
2. Describe why the code works the way it does.
3. Provide a simple, practical example of how the code could be used.
Your explanations should be easy to follow for someone new to programming and avoid unnecessary technical jargon.
"""

# user prompt：把 question 嵌进模板，明确要求「解释 + 使用示例」
user_prompt = f"""
Please explain the following code clearly and give an example of its use:
```
{question}
```
"""


In [5]:
# ========== 流式回答：统一封装「某个 client + 某个 model」的打字机显示 ==========

# 定义流式函数：传入 OpenAI 兼容 client 与模型名，边收边刷新 Markdown
def stream_response(client, model):
    # 发起 Chat Completions；stream=True 让服务端按块推送，而不是等整段完成
    response = client.chat.completions.create(
        model=model,
        messages=[
            # system：定「怎么答」；user：放具体题目
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )

    # 先放一个空的 Markdown 占位，拿到 display_id，后面才能原地更新（不会刷出一堆新格子）
    display_handle = display(Markdown(""), display_id=True)
    # 累加到目前为止收到的全部文本
    full_response = ""
    # 遍历流式 chunk：每个 chunk 可能带一小段 delta.content
    for chunk in response:
        # 取出本块新增文本；有的 chunk 没有 content，用 or '' 避免 None
        response_text = chunk.choices[0].delta.content or ''
        # 拼到完整答复上
        full_response += response_text
        # 用同一个 display_id 刷新 Markdown，实现「边生成边渲染」
        update_display(Markdown(full_response), display_id=display_handle.display_id)


In [6]:
# ========== 路径 A：云端 OpenAI（MODEL_GPT）流式解释 ==========

# 使用默认 openai 客户端 + 环境变量里的 MODEL_GPT（默认 gpt-4o-mini）
stream_response(openai, MODEL_GPT)


Sure! Let's break down the code you provided into smaller pieces to understand what it does and why it behaves in that way.

### Step-by-Step Explanation

1. **Understanding `books`:**  
   The code assumes that there is a variable called `books`, which is likely a list (or collection) of dictionaries. Each dictionary represents a book and may contain various pieces of information about that book, such as its title, author, year of publication, etc.

2. **List Comprehension with a Conditional Statement:**  
   The part `{book.get("author") for book in books if book.get("author")}` is a set comprehension:
   - `for book in books`: This means we are going to look at each `book` in the `books` collection, one at a time.
   - `book.get("author")`: This tries to get the value associated with the key "author" from the current `book` dictionary. The `get()` method is used here, which is safe; if "author" does not exist, it will return `None` instead of causing an error.
   - `if book.get("author")`: This part checks if the author exists (i.e., it's not `None`). If there is an author, then it continues to the next step.
   - The curly braces `{}` around the comprehension mean we are creating a set. A set is a collection of unique items; it automatically removes duplicates.

3. **Using `yield from`:**  
   The `yield from` expression is used within a generator function. A generator function produces values one at a time and can be iterated over. The `yield from` statement is used to yield all values from another iterable (in this case, our set of authors). This means that when this line of code runs, it will return each unique author found in the `books` collection.

### Why the Code Works This Way
- The code is structured to gather all unique authors from a list of books. By using a set comprehension, it filters out any books that don’t have an author, ensuring that only valid authors are collected.
- The `yield from` allows the authors to be generated one at a time, which is memory efficient, especially if there are many authors and it is not necessary to keep them all in memory at once.

### Practical Example
Suppose you have the following list of books, which includes authors:

```python
books = [
    {"title": "Book A", "author": "Author 1"},
    {"title": "Book B", "author": "Author 2"},
    {"title": "Book C", "author": None},
    {"title": "Book D"},
    {"title": "Book E", "author": "Author 1"}
]

def unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Now we can use our generator function to get unique authors
for author in unique_authors(books):
    print(author)
```

### Expected Output
When you run the above code, the output will be:
```
Author 1
Author 2
```

### Summary
In this example, the code effectively retrieves unique authors from a structured collection of books. It handles missing author information gracefully and returns each unique author when iterated over. This type of functionality could be useful in applications like a library system, book catalog, or any environment where authors need to be listed without duplicates.

In [7]:
# ========== 路径 B：本地 Ollama（MODEL_LLAMA）流式解释 ==========

# 同一套 stream_response；换 ollama 客户端与本地模型名（需 Ollama 在跑且已 pull）
stream_response(ollama, MODEL_LLAMA)


I'd be happy to help you understand this code.

**What does the code do?**

The code uses a concept called "yield from" which is used with generators. Let me break it down step by step:

1. `{book.get("author") for book in books`: This is a **generator expression** that produces an iterable sequence of values. In this case, it's using the `get` method to retrieve a value ("author") from each `book` object.
2. `if book.get("author")`: This adds a condition to the generator expression - only include the author value if the "author" key exists in the `book` dictionary and its value is not empty or null.
3. `yield from ...`: This tells Python to use the values returned by the generator expression within the outer scope, rather than trying to process them as individual iterable objects.

So, when put together, this code **yields** an author name from each book object that has an "author" key.

**Why does it work this way?**

The `yield from` keywords allows us to skip one level of iteration and use the values produced by another generator expression. In Python, a generator is a special type of iterable that produces values on-the-fly, rather than building them up in memory all at once.

By using `yield from`, we're essentially saying "take the output from this inner generator expression (a sequence of author names) and yield it **at this point**, without having to process each value individually".

**Example use case**

Suppose you have a list of book objects, like so:
```python
books = [
    {"title": "Book 1", "author": "John Doe"},
    {"title": "Book 2", "author": None},  # no author here!
    {"title": "Book 3", "author": "Jane Smith"}
]
```
If you use the code with these books:
```python
authors = yield from {book.get("author") for book in books if book.get("author")}
print(authors)
```
The output will be a list of author names: `['John Doe', 'Jane Smith']`

However, "Book 2" is skipped because its "author" key has no value.

In this example, using `yield from` makes the code more concise and efficient, since we're directly producing the desired sequence of authors without having to process them individually.

In [8]:
# ========== 路径 C：同一端点上的 cloud 模型名（MODEL_LLAMA_CLOUD） ==========

# 仍走本地 Ollama /v1 端点，但 model= 换成 OLLAMA_CLOUD_MODEL；用于对比不同后端的风格与能力
stream_response(ollama_cloud, MODEL_LLAMA_CLOUD)


### What the line does

```python
yield from {book.get("author") for book in books if book.get("author")}
```

1. **`for book in books`** –  
   The code goes through every element in the iterable called **`books`**.  
   We assume each element is a *dictionary* that may contain a key named `"author"`.

2. **`book.get("author")`** –  
   `dict.get(key)` returns the value associated with **`key`**.  
   If the key does not exist, it returns `None` (or any other default you pass).  
   So `book.get("author")` fetches the author of the current book, or `None` if that book has no author.

3. **`if book.get("author")`** –  
   This is a filter.  
   It keeps only the books whose author value is *truthy*.  
   `"Alice"` is truthy, but `None`, `""`, `0`, etc. are *falsey* and are discarded.

4. **`{ … }`** –  
   Curly braces create a **set comprehension**.  
   The expression inside the braces builds a *set* – an unordered collection that automatically removes duplicates.  
   So the result is a set of all distinct author names found in the books that passed the filter.

5. **`yield from …`** –  
   `yield from` takes an *iterable* (here, the set of authors) and yields each element of that iterable one by one.  
   Inside a generator function, it is a concise way to “return all the values” of the set without writing an explicit loop.

In short:  
The line builds a set of unique, non‑null author names from `books` and then yields each author to the caller of the generator.

---

### Why the code works the way it does

| Step | What happens | Why it’s useful |
|------|--------------|-----------------|
| **Set comprehension** | Creates a set while iterating | Removes duplicate authors automatically, so you don’t get repeats. |
| **`dict.get()`** | Safely accesses `"author"` | If a book lacks the key, it returns `None` instead of raising a `KeyError`. |
| **Filter `if book.get("author")`** | Skips books with no author | Prevents `None` or empty strings from ending up in the result. |
| **`yield from`** | Yields all set items | It’s a clean shortcut for: `for a in set: yield a`. |

**Important nuance**:  
Because a set is *unordered*, the authors will come out in an unpredictable order.  
If you need to preserve the order in which authors first appeared, use a list or an `OrderedDict` instead.

---

### Practical example

Below is a small generator function that uses the line in question, followed by a demonstration of its output.

```python
# The generator that yields all unique authors
def unique_authors(books):
    # Builds a set of author names that are not None/empty,
    # then yields each one.
    yield from {book.get("author") for book in books if book.get("author")}

# Example data: a list of book dictionaries
books = [
    {"title": "The Great Adventure", "author": "Alice"},
    {"title": "Mystery Night",      "author": "Bob"},
    {"title": "Lost in Time",      "author": None},   # No author – will be skipped
    {"title": "The Great Adventure", "author": "Alice"},  # Duplicate author
    {"title": "Unknown Journey",    "author": ""},    # Empty string – falsey, skipped
]

# Use the generator and print each author it yields
for author in unique_authors(books):
    print(author)
```

**Output (order may vary):**

```
Alice
Bob
```

*Explanation of the output*

1. `"Alice"` appears twice but the set keeps only one copy.  
2. `"Bob"` appears once.  
3. The book with `author: None` and the one with an empty string are filtered out by the `if` clause.

---

### Take‑away

- **Set comprehension** builds a collection of unique values in one line.  
- **`book.get("author")`** safely extracts the author; if missing, it returns `None`.  
- The **`if` filter** removes any falsey (missing or empty) authors.  
- **`yield from`** turns that set into a stream of values produced by a generator.

You can drop this line into any generator function where you want to export a list of distinct, non‑null strings extracted from a sequence of dictionaries.